<a href="https://colab.research.google.com/github/Lourdes-Churacutipa/Repositorio-de-lourdes/blob/main/Examen_parcial_Lourdes_Churacutipa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Examen Parcial**
## **Lourdes Churacutipa Llanqui**
**IDE:** (Google Colab o VSCode)
**Fecha:** Mayo 11, 2026

**Puntaje total:** 8 puntos

**Instrucciones:**
- Completa los ejercicios en esta notebook
- Escribe los códigos en las celdas provistas
- Sube tu archivo `.ipynb` completado a Canvas

**Integridad Académica:** Este es un examen individual. Puedes consultar los materiales del curso y documentación de Python, pero **no** podrás usar herramientas de IA. El código debe ser tuyo.

In [16]:
# Ejecuta esta celda primero
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from io import StringIO
print("Configuración completa.")

Configuración completa.


---
## Problema 1: Procesador de Nombres de Archivos Landsat (4 puntos)

Trabajas con imágenes satelitales Landsat del Perú. Los nombres siguen el formato:

```
LC08_L2SP_008067_20240615_02_T1_SR_B4.TIF
```
Componentes: `{sensor}_{nivel}_{path_row}_{fecha}_{coleccion}_{tier}_SR_{banda}.TIF`

> Los paths 003-009, filas 062-071 cubren el territorio peruano (Madre de Dios, Loreto, Lima, Cusco).

**Parte A (2 pts):** Escribe la función `procesar_nombre_landsat(nombre_archivo)` que devuelva un diccionario con `sensor`, `path`, `row`, `fecha` (formato `"AAAA-MM-DD"`) y `banda`.

**Parte B (1 pt):** Escribe la función `clasificar_banda(banda)` que devuelva el nombre según:

| Banda | Nombre |
|-------|--------|
| B1 | Aerosol costero |
| B2 | Azul |
| B3 | Verde |
| B4 | Rojo |
| B5 | Infrarrojo cercano (NIR) |
| B6 | SWIR1 |
| B7 | SWIR2 |

Si la banda no está en la tabla, devuelve `"Desconocida"`.

**Parte C (1 pt):** Procesa la lista de archivos: imprime un resumen por archivo (fecha / path / row / nombre de banda) y cuenta cuántas fechas únicas hay.

In [17]:
# Archivos Landsat sobre el Perú (paths 008-009: Madre de Dios, Ucayali, Loreto)
archivos = [
    "LC08_L2SP_008067_20240615_02_T1_SR_B4.TIF",
    "LC08_L2SP_008067_20240615_02_T1_SR_B5.TIF",
    "LC08_L2SP_008067_20240701_02_T1_SR_B3.TIF",
    "LC08_L2SP_009067_20240615_02_T1_SR_B4.TIF",
    "LC09_L2SP_008067_20240708_02_T1_SR_B6.TIF",
    "LC08_L2SP_008067_20240701_02_T1_SR_B4.TIF",
]

# Parte A: función procesar_nombre_landsat
def procesar_nombre_landsat(nombre_archivo):
    """
    Procesa el nombre de un archivo Landsat y extrae un diccionario con:
    sensor, path, row, fecha (YYYY-MM-DD) y banda.
    """
    # Eliminar extensión .TIF
    # separar los componentes por '_'
    nombre_base = nombre_archivo.replace(".TIF", "")
    partes = nombre_base.split("_")

    sensor = partes[0]
    path_row = partes[2]
    path = path_row[:3]
    row = path_row[3:]

    fecha_raw = partes[3]
    fecha = f"{fecha_raw[:4]}-{fecha_raw[4:6]}-{fecha_raw[6:]}"

    banda = partes[7]

    return {
        "sensor": sensor,
        "path": path,
        "row": row,
        "fecha": fecha,
        "banda": banda
    }

# Parte B: función clasificar_banda
def clasificar_banda(banda):
    """
    Devuelve el nombre descriptivo de la banda Landsat.
    """
    bandas_dict = {
        "B1": "Aerosol costero",
        "B2": "Azul",
        "B3": "Verde",
        "B4": "Rojo",
        "B5": "Infrarrojo cercano (NIR)",
        "B6": "SWIR1",
        "B7": "SWIR2"
    }
    return bandas_dict.get(banda, "Desconocida")

# Parte C: procesar todos los archivos e imprimir resultados
fechas_unicas = set()

print("--- RESUMEN DE ARCHIVOS LANDSAT ---")
for archivo in archivos:
    info = procesar_nombre_landsat(archivo)
    nombre_banda = clasificar_banda(info["banda"])

    ## Agregar fecha al conjunto de fechas únicas
    fechas_unicas.add(info["fecha"])

    # Imprimir resumen de cada archivo
    print(f"Fecha: {info['fecha']} | Path: {info['path']} | Row: {info['row']} | Banda: {info['banda']} ({nombre_banda})")

print("-" * 50)
print(f"Total de fechas únicas encontradas: {len(fechas_unicas)}")


--- RESUMEN DE ARCHIVOS LANDSAT ---
Fecha: 2024-06-15 | Path: 008 | Row: 067 | Banda: B4 (Rojo)
Fecha: 2024-06-15 | Path: 008 | Row: 067 | Banda: B5 (Infrarrojo cercano (NIR))
Fecha: 2024-07-01 | Path: 008 | Row: 067 | Banda: B3 (Verde)
Fecha: 2024-06-15 | Path: 009 | Row: 067 | Banda: B4 (Rojo)
Fecha: 2024-07-08 | Path: 008 | Row: 067 | Banda: B6 (SWIR1)
Fecha: 2024-07-01 | Path: 008 | Row: 067 | Banda: B4 (Rojo)
--------------------------------------------------
Total de fechas únicas encontradas: 3


---
## Problema 2: Inventario Forestal en Madre de Dios (4 puntos)

El **SERFOR** realiza inventarios forestales en Madre de Dios. Los datos incluyen valores faltantes (`-999`) y mediciones con posibles errores.

**Parte A (1 pt):** Escribe `calcular_area_basal(dap_cm)` que devuelva el área basal en m²: $AB = \pi \times (DAP/200)^2$, o `None` si DAP ≤ 0 o DAP == -999.

**Parte B (1 pt):** Escribe `clasificar_arbol(dap_cm, altura_m)` que devuelva un diccionario con:
- `clase`: `"Brinzal"` (<10 cm), `"Latizal"` (10-25 cm), `"Fustal menor"` (25-50 cm), `"Fustal mayor"` (≥50 cm)
- `alerta`: `True` si DAP > 200 cm, altura > 60 m, o altura < 1 m con DAP > 10 cm

**Parte C (2 pts):** Procesa los datos: calcula área basal y clasifica cada árbol, omite filas con -999, imprime advertencia para alertas, y calcula: total de árboles válidos, área basal total, conteo por clase y número de alertas.

In [18]:
import math

# Inventario forestal - Madre de Dios, Perú (datos SERFOR)
# Formato: [id, especie, dap_cm, altura_m]
datos_arboles = [
    [1,  "Swietenia macrophylla",    35.4, 22.1],   # Caoba
    [2,  "Cedrela odorata",          28.2, 18.5],   # Cedro
    [3,  "Cedrelinga cateniformis", -999,  25.0],   # Tornillo - DAP faltante
    [4,  "Virola surinamensis",      18.7, 12.3],   # Cumala
    [5,  "Dipteryx micrantha",       52.1, 24.8],   # Shihuahuaco
    [6,  "Calycophyllum spruceanum",  8.5,  6.2],   # Capirona
    [7,  "Terminalia oblonga",       45.0, 85.0],   # Yacushapana - altura sospechosa
    [8,  "Cedrelinga cateniformis",  62.3, 28.4],   # Tornillo
    [9,  "Swietenia macrophylla",    41.2, -999],   # Caoba - altura faltante
    [10, "Hura crepitans",           22.5,  0.5],   # Catahua - sospechoso
    [11, "Schizolobium parahybum",    5.2,  3.1],   # Pino chuncho
    [12, "Guazuma crinita",          38.9, 21.7],   # Bolaina
]

# Parte A: función calcular_area_basal
def calcular_area_basal(dap_cm):
    """
    Calcula el área basal en m²: AB = pi * (DAP / 200)^2
    Devuelve None si DAP <= 0 o DAP == -999.
    """
    if dap_cm is None or dap_cm <= 0 or dap_cm == -999:
        return None
    return math.pi * ((dap_cm / 200) ** 2)

# Parte B: función clasificar_arbol
def clasificar_arbol(dap_cm, altura_m):
    """
    Clasifica el árbol según su DAP y evalúa si tiene alertas de error en la medición.
    """
    # #Determinación de la clase
    if dap_cm < 10:
        clase = "Brinzal"
    elif 10 <= dap_cm < 25:
        clase = "Latizal"
    elif 25 <= dap_cm < 50:
        clase = "Fustal menor"
    else:  # dap_cm >= 50
        clase = "Fustal mayor"

    ## Evaluador de alerta de inconsistencia
    alerta = False
    if dap_cm > 200 or altura_m > 60 or (altura_m < 1 and dap_cm > 10):
        alerta = True

    return {
        "clase": clase,
        "alerta": alerta
    }

# Parte C: procesar datos e imprimir resultados
arboles_validos = 0
area_basal_total = 0.0
conteo_clases = {"Brinzal": 0, "Latizal": 0, "Fustal menor": 0, "Fustal mayor": 0}
total_alertas = 0

print("--- PROCESAMIENTO DE REGISTROS FORESTALES ---")

for arbol in datos_arboles:
    id_arbol, especie, dap_cm, altura_m = arbol

    # Omitir filas con valores faltantes (-999) en DAP o altura
    if dap_cm == -999 or altura_m == -999:
        print(f"[OMITIDO] Árbol ID {id_arbol} ({especie}): Datos faltantes (-999).")
        continue

    ## paea calcular área basal y clasificación
    ab = calcular_area_basal(dap_cm)
    clasif = clasificar_arbol(dap_cm, altura_m)

    ### para acumular estadísticos
    arboles_validos += 1
    area_basal_total += ab
    conteo_clases[clasif["clase"]] += 1

    ### verificamos la alerta
    if clasif["alerta"]:
        total_alertas += 1
        print(f"[ADVERTENCIA] Árbol ID {id_arbol} ({especie}) presenta mediciones sospechosas "
              f"(DAP: {dap_cm} cm, Altura: {altura_m} m).")

print("\n" + "=" * 50)
print("--- RESUMEN FINAL DE RESULTADOS ---")
print(f"Total de árboles válidos: {arboles_validos}")
print(f"Área basal total: {area_basal_total:.4f} m²")
print("Conteo de árboles por clase:")
for clase, cantidad in conteo_clases.items():
    print(f"  - {clase}: {cantidad}")
print(f"Número total de alertas emitidas: {total_alertas}")


--- PROCESAMIENTO DE REGISTROS FORESTALES ---
[OMITIDO] Árbol ID 3 (Cedrelinga cateniformis): Datos faltantes (-999).
[ADVERTENCIA] Árbol ID 7 (Terminalia oblonga) presenta mediciones sospechosas (DAP: 45.0 cm, Altura: 85.0 m).
[OMITIDO] Árbol ID 9 (Swietenia macrophylla): Datos faltantes (-999).
[ADVERTENCIA] Árbol ID 10 (Hura crepitans) presenta mediciones sospechosas (DAP: 22.5 cm, Altura: 0.5 m).

--- RESUMEN FINAL DE RESULTADOS ---
Total de árboles válidos: 10
Área basal total: 1.0318 m²
Conteo de árboles por clase:
  - Brinzal: 2
  - Latizal: 2
  - Fustal menor: 4
  - Fustal mayor: 2
Número total de alertas emitidas: 2


---
## Problema 3: Calidad de Agua en Ríos Andino-Amazónicos (4 puntos)

Analizas datos de calidad de agua de tres ríos monitoreados por la **Autoridad Nacional del Agua (ANA)**: Ucayali (selva baja), Tambopata (selva alta) y Mantaro (sierra).

**Parte A (1 pt):** Carga el CSV en un DataFrame. Muestra forma, tipos, primeras filas y valores faltantes. Convierte `fecha` a datetime.

**Parte B (2 pts):**
1. Calcula media, std, mínimo y máximo de `od_mg_l` por `estacion_id`. ¿Cuál estación tiene el OD más bajo?
2. Crea columna `estado_od`: `"Crítico"` (OD < 4), `"Bajo"` (4-6), `"Adecuado"` (6-8), `"Bueno"` (≥ 8). Cuenta por estación.

**Parte C (1 pt):** Filtra donde temperatura > 20°C Y pH entre 6.5-8.5. Calcula conductividad media por mes. Identifica la combinación estación-mes con más lecturas "Crítico" o "Bajo".

In [8]:
# Dataset calidad de agua - ríos andino-amazónicos del Perú (ANA)
calidad_agua_csv = (
    "estacion_id,fecha,temp_c,od_mg_l,ph,conductividad_us\n"
    "RIO_UCAYALI,2024-05-15,24.3,7.2,7.1,145\n"
    "RIO_UCAYALI,2024-05-22,25.1,6.8,7.0,152\n"
    "RIO_UCAYALI,2024-06-05,24.8,6.5,6.9,158\n"
    "RIO_UCAYALI,2024-06-19,25.5,5.8,6.8,165\n"
    "RIO_UCAYALI,2024-07-03,24.2,5.2,7.0,172\n"
    "RIO_UCAYALI,2024-07-17,23.8,4.8,7.1,168\n"
    "RIO_UCAYALI,2024-08-01,24.5,4.2,7.2,175\n"
    "RIO_UCAYALI,2024-08-15,25.2,5.0,7.0,169\n"
    "RIO_TAMBOPATA,2024-05-15,21.8,8.5,7.4,98\n"
    "RIO_TAMBOPATA,2024-05-22,22.5,8.1,7.5,105\n"
    "RIO_TAMBOPATA,2024-06-05,22.9,7.8,7.3,112\n"
    "RIO_TAMBOPATA,2024-06-19,23.4,7.2,7.2,118\n"
    "RIO_TAMBOPATA,2024-07-03,22.1,6.8,7.1,125\n"
    "RIO_TAMBOPATA,2024-07-17,21.8,6.5,7.0,121\n"
    "RIO_TAMBOPATA,2024-08-01,22.5,6.9,7.1,128\n"
    "RIO_TAMBOPATA,2024-08-15,23.1,7.2,7.2,115\n"
    "RIO_MANTARO,2024-05-15,13.1,9.5,6.5,312\n"
    "RIO_MANTARO,2024-05-22,14.2,8.8,6.4,325\n"
    "RIO_MANTARO,2024-06-05,12.8,8.2,6.3,338\n"
    "RIO_MANTARO,2024-06-19,13.5,7.5,6.2,352\n"
    "RIO_MANTARO,2024-07-03,11.9,6.8,6.0,368\n"
    "RIO_MANTARO,2024-07-17,10.8,5.9,5.9,378\n"
    "RIO_MANTARO,2024-08-01,11.5,5.2,6.1,385\n"
    "RIO_MANTARO,2024-08-15,12.3,4.8,6.2,372\n"
)
# Pista: usa pd.read_csv(StringIO(calidad_agua_csv))

In [9]:
# Parte A: Carga y explora los datos

# Cargar datos CSV usando StringIO
df_agua = pd.read_csv(StringIO(calidad_agua_csv))

# Mostrar forma y tipos de datos
print("Forma del DataFrame (filas, columnas):", df_agua.shape)
print("\nTipos de datos original:")
print(df_agua.dtypes)

# Convertir la columna fecha a tipo datetime
df_agua['fecha'] = pd.to_datetime(df_agua['fecha'])

# Verificar valores faltantes por columna
print("\nValores faltantes por columna:")
print(df_agua.isnull().sum())

# Mostrar las primeras filas
print("\nPrimeras 5 filas del DataFrame:")
print(df_agua.head())

Forma del DataFrame (filas, columnas): (24, 6)

Tipos de datos original:
estacion_id          object
fecha                object
temp_c              float64
od_mg_l             float64
ph                  float64
conductividad_us      int64
dtype: object

Valores faltantes por columna:
estacion_id         0
fecha               0
temp_c              0
od_mg_l             0
ph                  0
conductividad_us    0
dtype: int64

Primeras 5 filas del DataFrame:
   estacion_id      fecha  temp_c  od_mg_l   ph  conductividad_us
0  RIO_UCAYALI 2024-05-15    24.3      7.2  7.1               145
1  RIO_UCAYALI 2024-05-22    25.1      6.8  7.0               152
2  RIO_UCAYALI 2024-06-05    24.8      6.5  6.9               158
3  RIO_UCAYALI 2024-06-19    25.5      5.8  6.8               165
4  RIO_UCAYALI 2024-07-03    24.2      5.2  7.0               172


In [19]:
# Parte B: Análisis con agrupación y clasificación OD

###Estadísticas descriptivas de od_mg_l por estación
stats_od = df_agua.groupby('estacion_id')['od_mg_l'].agg(['mean', 'std', 'min', 'max'])
print("Estadísticas descriptivas de Oxígeno Disuelto (OD) por estación:")
print(stats_od)

## identificar la estación con el OD mínimo más bajo
estacion_od_minimo = stats_od['min'].idxmin()
valor_min = stats_od['min'].min()
print(f"\nLa estación con el OD más bajo es: {estacion_od_minimo} (Mínimo: {valor_min} mg/L)")

#### Categorización de OD mediante función auxiliar/cut
def categorizar_od(od):
    if od < 4:
        return "Crítico"
    elif 4 <= od < 6:
        return "Bajo"
    elif 6 <= od < 8:
        return "Adecuado"
    else:  # od >= 8
        return "Bueno"

df_agua['estado_od'] = df_agua['od_mg_l'].apply(categorizar_od)

# Conteo de categorías por estación
print("\nConteo del estado de OD por estación:")
conteo_estado = pd.crosstab(df_agua['estacion_id'], df_agua['estado_od'])
print(conteo_estado)

Estadísticas descriptivas de Oxígeno Disuelto (OD) por estación:
                 mean       std  min  max
estacion_id                              
RIO_MANTARO    7.0875  1.709166  4.8  9.5
RIO_TAMBOPATA  7.3750  0.692305  6.5  8.5
RIO_UCAYALI    5.6875  1.062931  4.2  7.2

La estación con el OD más bajo es: RIO_UCAYALI (Mínimo: 4.2 mg/L)

Conteo del estado de OD por estación:
estado_od      Adecuado  Bajo  Bueno
estacion_id                         
RIO_MANTARO           2     3      3
RIO_TAMBOPATA         6     0      2
RIO_UCAYALI           3     5      0


In [20]:
# Parte C: Filtrado y resumen
## Filtrar registros donde temperatura > 20°C Y pH entre 6.5 y 8.5
filtro_temp_ph = (df_agua['temp_c'] > 20) & (df_agua['ph'] >= 6.5) & (df_agua['ph'] <= 8.5)
df_filtrado = df_agua[filtro_temp_ph].copy()

# Conductividad media por mes
df_filtrado['mes'] = df_filtrado['fecha'].dt.strftime('%Y-%m')
cond_media_mes = df_filtrado.groupby('mes')['conductividad_us'].mean()
print("Conductividad media por mes (en datos filtrados):")
print(cond_media_mes)

# Combinación estación-mes con más lecturas "Crítico" o "Bajo"
# Crear columna de mes en el DataFrame completo
df_agua['mes'] = df_agua['fecha'].dt.strftime('%Y-%m')

# Filtrar lecturas vulnerables (Crítico o Bajo)
df_alerta = df_agua[df_agua['estado_od'].isin(['Crítico', 'Bajo'])]

# Conteo por combinación Estación - Mes
conteo_alertas = df_alerta.groupby(['estacion_id', 'mes']).size()

print("\nConteo de lecturas en estado 'Crítico' o 'Bajo' por combinación Estación-Mes:")
print(conteo_alertas)

# Identificar la combinación máxima
comb_max = conteo_alertas.idxmax()
num_alertas = conteo_alertas.max()
print(f"\nLa combinación estación-mes con más lecturas vulnerables es: {comb_max[0]} en {comb_max[1]} ({num_alertas} lecturas)")


Conductividad media por mes (en datos filtrados):
mes
2024-05    125.00
2024-06    138.25
2024-07    146.50
2024-08    146.75
Name: conductividad_us, dtype: float64

Conteo de lecturas en estado 'Crítico' o 'Bajo' por combinación Estación-Mes:
estacion_id  mes    
RIO_MANTARO  2024-07    1
             2024-08    2
RIO_UCAYALI  2024-06    1
             2024-07    2
             2024-08    2
dtype: int64

La combinación estación-mes con más lecturas vulnerables es: RIO_MANTARO en 2024-08 (2 lecturas)


---
## Problema 4: Comparación Estadística de Parcelas Forestales (4 puntos)

Investigadores del **INIA - Estación Experimental Pucallpa** midieron biomasa arbórea (kg) en parcelas control y parcelas con aprovechamiento forestal de impacto reducido (AFIR).

**Parte A (1.5 pts):** Calcula estadísticas descriptivas (media, std, mediana) por grupo. Realiza una prueba t de dos muestras independientes (α = 0.05). Plantea hipótesis nula y alternativa, reporta estadístico t y p-valor, y escribe una conclusión.

**Parte B (1.5 pts):** Calcula el coeficiente de correlación de Pearson entre DAP y biomasa para todo el dataset. Evalúa si es significativo (α = 0.05). Interpreta fuerza y dirección.

**Parte C (1 pt):** Ajusta una distribución normal a la biomasa de parcelas control. Reporta μ y σ. Calcula la probabilidad de biomasa > 150 kg. ¿Qué valor corresponde al percentil 90?

In [12]:
# Datos de parcelas forestales - INIA Pucallpa
np.random.seed(458)

n_control = 35
dap_control    = np.random.uniform(15, 50, n_control)
biomasa_control = 0.1 * dap_control**2.2 + np.random.normal(0, 15, n_control)
biomasa_control = np.maximum(biomasa_control, 10)

n_afir = 30
dap_afir    = np.random.uniform(18, 55, n_afir)
biomasa_afir = 0.12 * dap_afir**2.2 + np.random.normal(5, 18, n_afir)
biomasa_afir = np.maximum(biomasa_afir, 10)

bosque_df = pd.DataFrame({
    "dap_cm":      np.concatenate([dap_control, dap_afir]),
    "biomasa_kg":  np.concatenate([biomasa_control, biomasa_afir]),
    "tratamiento": ["Control"]*n_control + ["AFIR"]*n_afir
})
print(bosque_df.head())

      dap_cm  biomasa_kg tratamiento
0  43.208835  395.956074     Control
1  49.475851  521.291644     Control
2  19.647405   54.799539     Control
3  23.651882  119.264060     Control
4  40.431164  335.576313     Control


In [21]:
# Parte A: Estadísticas descriptivas + prueba t

#$ Estadísticas descriptivas por tratamiento
stats_bosque = bosque_df.groupby('tratamiento')['biomasa_kg'].agg(
    Media='mean',
    Desviacion_Estandar='std',
    Mediana='median'
)
print("--- ESTADÍSTICAS DESCRIPTIVAS DE BIOMASA (kg) ---")
print(stats_bosque)

## Separación de grupos
biomasa_control = bosque_df[bosque_df['tratamiento'] == 'Control']['biomasa_kg']
biomasa_afir = bosque_df[bosque_df['tratamiento'] == 'AFIR']['biomasa_kg']

## Prueba t para muestras independientes (α = 0.05)
t_stat, p_val = stats.ttest_ind(biomasa_afir, biomasa_control)

print("\n--- PRUEBA T DE DOS MUESTRAS INDEPENDIENTES ---")
print("Hipótesis Nula (H0): No existe diferencia significativa en la biomasa promedio entre los tratamientos Control y AFIR.")
print("Hipótesis Alternativa (H1): Existe una diferencia significativa en la biomasa promedio entre los tratamientos Control y AFIR.")
print(f"Estadístico t: {t_stat:.4f}")
print(f"p-valor: {p_val:.4e}")

alpha = 0.05
if p_val < alpha:
    print(f"Conclusión: Dado que el p-valor ({p_val:.4e}) es menor a α = {alpha}, se rechaza H0.")
    print("Existe evidencia estadística significativa de una diferencia en la biomasa promedio entre los paramentos Control y AFIR.")
else:
    print(f"Conclusión: Dado que el p-valor ({p_val:.4e}) es mayor o igual a α = {alpha}, no se rechaza H0.")
    print("No existe evidencia estadística suficiente para afirmar una diferencia en la biomasa promedio entre ambos tratamientos.")


--- ESTADÍSTICAS DESCRIPTIVAS DE BIOMASA (kg) ---
                  Media  Desviacion_Estandar     Mediana
tratamiento                                             
AFIR         325.418953           213.516434  239.121336
Control      269.642640           158.210017  232.870089

--- PRUEBA T DE DOS MUESTRAS INDEPENDIENTES ---
Hipótesis Nula (H0): No existe diferencia significativa en la biomasa promedio entre los tratamientos Control y AFIR.
Hipótesis Alternativa (H1): Existe una diferencia significativa en la biomasa promedio entre los tratamientos Control y AFIR.
Estadístico t: 1.2070
p-valor: 2.3194e-01
Conclusión: Dado que el p-valor (2.3194e-01) es mayor o igual a α = 0.05, no se rechaza H0.
No existe evidencia estadística suficiente para afirmar una diferencia en la biomasa promedio entre ambos tratamientos.


In [22]:
# Parte B: Correlación de Pearson DAP vs. biomasa

#$Coeficiente de correlación de Pearson para todo el dataset
r_stat, p_val_corr = stats.pearsonr(bosque_df['dap_cm'], bosque_df['biomasa_kg'])

print("--- ANÁLISIS DE CORRELACIÓN DE PEARSON ---")
print(f"Coeficiente de correlación (r): {r_stat:.4f}")
print(f"p-valor: {p_val_corr:.4e}")

## Evaluación de significancia e interpretación
alpha = 0.05
if p_val_corr < alpha:
    print(f"\nEvaluación: La correlación es estadísticamente significativa (p < {alpha}).")
else:
    print(f"\nEvaluación: La correlación NO es estadísticamente significativa (p >= {alpha}).")

## Interpretación de la fuerza y dirección
if r_stat > 0.7:
    fuerza_direccion = "positiva muy fuerte"
elif r_stat > 0.5:
    fuerza_direccion = "positiva moderada-fuerte"
elif r_stat > 0:
    fuerza_direccion = "positiva débil"
else:
    fuerza_direccion = "negativa"

print(f"Interpretación: Existe una relación lineal {fuerza_direccion} entre el DAP y la biomasa, lo que indica que a mayor diámetro del árbol, mayor es la biomasa acumulada.")

--- ANÁLISIS DE CORRELACIÓN DE PEARSON ---
Coeficiente de correlación (r): 0.9565
p-valor: 2.0976e-35

Evaluación: La correlación es estadísticamente significativa (p < 0.05).
Interpretación: Existe una relación lineal positiva muy fuerte entre el DAP y la biomasa, lo que indica que a mayor diámetro del árbol, mayor es la biomasa acumulada.


In [23]:
# Parte C: Ajuste de distribución normal
## Ajuste de distribución normal a la biomasa de parcelas Control
mu, std = stats.norm.fit(biomasa_control)

print("--- AJUSTE DE DISTRIBUCIÓN NORMAL (Grupo Control) ---")
print(f"Media estimada (μ): {mu:.4f} kg")
print(f"Desviación estándar estimada (σ): {std:.4f} kg")

#$$ probabilidad de biomasa > 150 kg: P(X > 150) = 1 - P(X <= 150)
prob_mayor_150 = 1 - stats.norm.cdf(150, loc=mu, scale=std)
print(f"\nProbabilidad de biomasa > 150 kg: {prob_mayor_150 * 100:.2f}% (o {prob_mayor_150:.4f})")

# Percentil 90 (valor bajo el cual está el 90% de los datos)
percentil_90 = stats.norm.ppf(0.90, loc=mu, scale=std)
print(f"Valor correspondiente al percentil 90: {percentil_90:.4f} kg")

--- AJUSTE DE DISTRIBUCIÓN NORMAL (Grupo Control) ---
Media estimada (μ): 269.6426 kg
Desviación estándar estimada (σ): 155.9335 kg

Probabilidad de biomasa > 150 kg: 77.85% (o 0.7785)
Valor correspondiente al percentil 90: 469.4795 kg
